In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))


# ── shared/mlfp05/ex_2.py ──
"""
Shared utilities for MLFP05 Exercise 2 — CNNs on CIFAR-10.

Common infrastructure used by all technique files:
  - CIFAR-10 data loading and normalisation
  - Device detection and precision settings
  - ExperimentTracker / ModelRegistry setup
  - LightningModule wrapper for training
  - Visualisation helpers
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pytorch_lightning as pl
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.engines.model_registry import LocalFileArtifactStore
from kailash_ml.types import MetricSpec

# ── Environment & Reproducibility ────────────────────────────────────
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42, workers=True)

DEVICE = get_device()
# Notebook-safe path resolution: __file__ is undefined when this module is
# inlined into a Colab notebook by scripts/generate_selfcontained_notebook.py.
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
    ARTIFACT_DIR = _HERE.parent
except NameError:
    REPO_ROOT = Path.cwd()
    ARTIFACT_DIR = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"

# The ModelRegistry stores artifact files (model.pkl, model.onnx) in an
# ArtifactStore. We construct the store explicitly so exercises can attach
# an ONNX serving artifact to a registered version (see attach_onnx_artifact).
ARTIFACT_STORE = LocalFileArtifactStore(".kailash_ml/artifacts")

N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

# ── CIFAR-10 class labels (used for visualisation and apply sections) ─
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Per-channel normalisation statistics (CIFAR-10 population)
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


# ═══════════════════════════════════════════════════════════════════════
# Data Loading
# ═══════════════════════════════════════════════════════════════════════
def load_cifar10() -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    DataLoader,
    DataLoader,
]:
    """Load and normalise the full CIFAR-10 dataset.

    Returns:
        X_train, y_train, X_val, y_val, train_loader, val_loader
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_train = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_train = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))],
        dtype=torch.long,
    )
    X_val = torch.stack([val_set[i][0] for i in range(len(val_set))])
    y_val = torch.tensor(
        [val_set[i][1] for i in range(len(val_set))],
        dtype=torch.long,
    )

    # Per-channel normalisation
    X_train = (X_train - CIFAR_MEAN) / CIFAR_STD
    X_val = (X_val - CIFAR_MEAN) / CIFAR_STD

    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    print(
        f"CIFAR-10: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, "
        f"classes={N_CLASSES}: {CLASS_NAMES}"
    )
    return X_train, y_train, X_val, y_val, train_loader, val_loader


# ═══════════════════════════════════════════════════════════════════════
# Kailash Engine Setup
# ═══════════════════════════════════════════════════════════════════════
async def setup_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry, has_registry
    """
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker and
    # ModelRegistry use incompatible _kml_model_versions schemas. Route them
    # to separate sqlite files until upstream fixes the conflict.
    db = f"sqlite:///{db_name}"
    registry_db_name = (
        db_name.replace(".db", "_registry.db")
        if db_name.endswith(".db")
        else db_name + "_registry.db"
    )
    registry_db = f"sqlite:///{registry_db_name}"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn, artifact_store=ARTIFACT_STORE)
    return conn, tracker, "m5_cnns", registry, True


def init_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Sync wrapper for setup_engines."""
    return asyncio.run(setup_engines(db_name))


# ═══════════════════════════════════════════════════════════════════════
# Lightning Training Infrastructure
# ═══════════════════════════════════════════════════════════════════════
class LitCNN(pl.LightningModule):
    """Lightning wrapper for any nn.Module classifier.

    Tracks per-epoch training loss and validation accuracy for later
    logging to ExperimentTracker and visualisation with ModelVisualizer.
    """

    def __init__(self, model: nn.Module, lr: float = 1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.train_losses: list[float] = []
        self.val_accs: list[float] = []
        self._batch_losses: list[float] = []
        self._val_correct = 0
        self._val_total = 0

    def training_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        loss = F.cross_entropy(logits, y)
        self._batch_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self._batch_losses:
            self.train_losses.append(float(np.mean(self._batch_losses)))
            self._batch_losses = []

    def validation_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        self._val_correct += int((logits.argmax(dim=-1) == y).sum().item())
        self._val_total += int(y.size(0))

    def on_validation_epoch_end(self):
        if self._val_total > 0:
            self.val_accs.append(self._val_correct / self._val_total)
            self._val_correct = 0
            self._val_total = 0

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)


def get_precision_setting() -> str:
    """Return the optimal Lightning precision string for the current backend.

    Routes through ``kailash_ml.device()`` so MPS / CUDA / ROCm / Intel XPU /
    CPU are all selected by the same policy the rest of the platform uses.
    BackendInfo.precision is the canonical answer ("16-mixed" on Apple MPS
    + Tensor-Core GPUs, "32" on older GPUs and CPU).
    """
    import kailash_ml as km

    backend = km.device()
    print(f"  {backend.backend} detected -- using {backend.precision} precision")
    return backend.precision


def get_accelerator() -> str:
    """Return the Lightning accelerator string for the current backend."""
    import kailash_ml as km

    return km.device().accelerator


PRECISION = get_precision_setting()
ACCELERATOR = get_accelerator()


async def train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Train a CNN with Lightning and log metrics to ExperimentTracker.

    Uses the kailash-ml 1.1.1 ``tracker.track(...)`` async context manager.
    On normal exit the run is marked FINISHED; on exception it is marked FAILED.
    """
    lit = LitCNN(model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "architecture": name,
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": str(BATCH_SIZE),
                "dataset_size": str(len(train_loader.dataset)),
                "precision": PRECISION,
                "accelerator": ACCELERATOR,
            }
        )

        trainer.fit(lit, train_loader, val_loader)

        for epoch_idx, loss in enumerate(lit.train_losses):
            await run.log_metric("train_loss", loss, step=epoch_idx + 1)
        for epoch_idx, acc in enumerate(lit.val_accs):
            await run.log_metric("val_accuracy", acc, step=epoch_idx + 1)

        await run.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    print(
        f"  [{name}] final train loss={lit.train_losses[-1]:.4f}  "
        f"val acc={lit.val_accs[-1]:.3f}"
    )
    return lit.train_losses, lit.val_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            lr,
            epochs,
        )
    )


# ═══════════════════════════════════════════════════════════════════════
# Model Registration
# ═══════════════════════════════════════════════════════════════════════
async def register_model_async(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Register a trained model with metrics in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="val_accuracy", value=final_acc),
            MetricSpec(name="epochs", value=float(epochs)),
            MetricSpec(name="batch_size", value=float(BATCH_SIZE)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={final_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Sync wrapper for register_model_async."""
    return asyncio.run(
        register_model_async(registry, name, model, final_loss, final_acc, epochs)
    )


# ═══════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ═══════════════════════════════════════════════════════════════════════
def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance.

    Delegates to the canonical factory in shared.mlfp05 — the P2
    experimental notice is acknowledged there.
    """

    return _canonical()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
    x_label: str = "Epoch",
    y_label: str = "Value",
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(
        metrics=metrics,
        x_label=x_label,
        y_label=y_label,
    )
    fig.write_html(str(output_path))
    print(f"  Saved plot: {output_path}")


def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters())


def denormalise_cifar(img_tensor: torch.Tensor) -> torch.Tensor:
    """Reverse CIFAR-10 normalisation for display.

    Args:
        img_tensor: shape (C, H, W) or (B, C, H, W), normalised

    Returns:
        Tensor with pixel values clipped to [0, 1]
    """
    if img_tensor.dim() == 3:
        mean = CIFAR_MEAN.squeeze(0)
        std = CIFAR_STD.squeeze(0)
    else:
        mean = CIFAR_MEAN
        std = CIFAR_STD
    return (img_tensor * std + mean).clamp(0, 1)


# ═══════════════════════════════════════════════════════════════════════
# Serving Helpers (ONNX export + InferenceServer)
# ═══════════════════════════════════════════════════════════════════════
IMAGE_SHAPE = (3, 32, 32)
N_PIXELS = 3 * 32 * 32


class FlatImageAdapter(nn.Module):
    """Wrap an image classifier so it accepts flat pixel rows.

    InferenceServer's ONNX runtime turns each request record into ONE row of
    a 2-D float array, so the exported graph must take ``(batch, 3072)``.
    This adapter reshapes rows back to ``(batch, 3, 32, 32)`` before calling
    the wrapped CNN. ``predict(X)`` is the method ``OnnxBridge.validate``
    calls to get the native (PyTorch) outputs for comparison.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x.reshape(-1, *IMAGE_SHAPE))

    def predict(self, X) -> np.ndarray:
        device = next(self.model.parameters()).device
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
            return self.forward(x).cpu().numpy()


def images_to_records(images: torch.Tensor) -> list[dict[str, float]]:
    """Turn a batch of images into InferenceServer request records.

    One record per image, one key per pixel. Keys are zero-padded so the
    record order is the pixel order the ONNX graph expects.
    """
    rows = images.reshape(len(images), -1).cpu().tolist()
    return [{f"px{j:04d}": float(v) for j, v in enumerate(row)} for row in rows]


def attach_onnx_artifact(name: str, version: int, onnx_path: Path) -> None:
    """Store an exported ONNX file as ``model.onnx`` for a registered version.

    ``ModelRegistry.register_model`` saves the training artifact as
    ``model.pkl``; ``InferenceServer.from_registry(..., runtime="onnx")``
    loads ``model.onnx`` from the same ArtifactStore.

    OnnxBridge writes large weights to a sibling ``<file>.onnx.data`` file
    (ONNX external data); storing only the ``.onnx`` bytes would serve a model
    without its weights, so it is stored as ONE self-contained protobuf.
    """
    import onnx

    model_proto = onnx.load(str(onnx_path))  # pulls in any .onnx.data weights
    asyncio.run(
        ARTIFACT_STORE.save(name, version, model_proto.SerializeToString(), "model.onnx")
    )


# ── shared/mlfp05/ex_6.py ──
"""
Shared infrastructure for Exercise 6 — Graph Neural Networks.

Contains: Cora dataset loading (with Karate Club fallback), graph
normalisation, train/val/test split, kailash-ml engine setup,
graph visualisation helpers, and training harness.

Technique-specific code (GCN, GAT, GraphSAGE layers) does NOT belong here.
"""

import asyncio
import copy
import pickle
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex6_gnns"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cora"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# GRAPH DATA LOADING
# ════════════════════════════════════════════════════════════════════════


def load_cora() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Cora — 2708 papers, 1433 bag-of-words features, 7 classes.

    Returns:
        X_np: node features (N, F)
        A_np: dense adjacency matrix (N, N)
        y_np: node labels (N,)
        edge_index_np: edge list (2, E) for link prediction
        dataset_name: "Cora"
        n_classes: 7
    """
    from torch_geometric.datasets import Planetoid

    dataset = Planetoid(root=str(DATA_DIR), name="Cora")
    # torch_geometric.data.Data has dynamic attributes (num_nodes/x/y/edge_index);
    # use Any to avoid type-checker false positives without losing runtime fidelity.
    data: Any = dataset[0]
    n = data.num_nodes
    X_np = data.x.numpy().astype(np.float32)
    y_np = data.y.numpy().astype(np.int64)

    # Build a dense adjacency matrix from the edge_index. Cora has ~10k
    # directed edges (5278 undirected) over 2708 nodes; the dense matrix
    # is ~7M entries which fits comfortably in CPU memory.
    A_np = np.zeros((n, n), dtype=np.float32)
    edge_index_np = data.edge_index.numpy()
    src = edge_index_np[0]
    dst = edge_index_np[1]
    A_np[src, dst] = 1.0
    A_np[dst, src] = 1.0  # symmetrise just in case
    n_classes = int(dataset.num_classes)
    return X_np, A_np, y_np, edge_index_np, "Cora", n_classes


def load_karate() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Zachary's Karate Club — 34 nodes, 78 edges, 2 factions."""
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    # weight=None: an unweighted 0/1 adjacency (the graph carries interaction
    # counts as edge weights, which would otherwise leak into A).
    A_np = nx.to_numpy_array(G, dtype=np.float32, weight=None)
    labels = np.array(
        [0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in range(n)],
        dtype=np.int64,
    )
    # Karate has no node features; use one-hot identity (transductive)
    X_np = np.eye(n, dtype=np.float32)
    # Build edge_index from adjacency
    src, dst = np.where(A_np > 0)
    edge_index_np = np.stack([src, dst]).astype(np.int64)
    return X_np, A_np, labels, edge_index_np, "Karate Club", 2


def normalise_adjacency(A: torch.Tensor) -> torch.Tensor:
    """Symmetric-normalised adjacency with self-loops: D^{-1/2} (A + I) D^{-1/2}.

    This is the GCN propagation matrix (Kipf & Welling, 2017). It is NOT
    the graph Laplacian — the normalised Laplacian is I - D^{-1/2} A D^{-1/2}.
    """
    A_hat = A + torch.eye(A.size(0), device=A.device)
    d_inv_sqrt = A_hat.sum(dim=1).pow(-0.5)
    d_inv_sqrt[torch.isinf(d_inv_sqrt)] = 0.0
    return d_inv_sqrt.unsqueeze(1) * A_hat * d_inv_sqrt.unsqueeze(0)


def load_graph_data() -> dict:
    """Load Cora (with Karate fallback) and return all graph tensors.

    Returns a dict with keys:
        X, A, y, A_norm, A_hat — torch tensors on device
        X_np, A_np, y_np, edge_index_np — numpy arrays
        train_mask, val_mask, test_mask — boolean masks on device
        N, F_dim, n_classes, n_edges_undirected, dataset_name — scalars
    """
    try:
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_cora()
    except Exception as exc:
        print(
            f"Could not load Cora ({type(exc).__name__}: {exc}); "
            "falling back to Karate Club"
        )
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_karate()

    N = X_np.shape[0]
    F_dim = X_np.shape[1]
    n_edges_undirected = int(A_np.sum() // 2)
    print(
        f"Graph: {dataset_name} — {N} nodes, {n_edges_undirected} undirected edges, "
        f"feature_dim={F_dim}, classes={n_classes}"
    )
    class_counts = ", ".join(f"c{c}={int((y_np == c).sum())}" for c in range(n_classes))
    print(f"  per-class counts: {class_counts}")

    X = torch.from_numpy(X_np).to(device)
    A = torch.from_numpy(A_np).to(device)
    y = torch.from_numpy(y_np).to(device)

    # Add self-loops and build the symmetric-normalised adjacency
    # D^{-1/2} (A + I) D^{-1/2}
    A_hat = A + torch.eye(N, device=device)
    A_norm = normalise_adjacency(A)

    # Train/val/test split — 20% train, 20% val, 60% test (per class)
    train_mask = torch.zeros(N, dtype=torch.bool, device=device)
    val_mask = torch.zeros(N, dtype=torch.bool, device=device)
    rng = np.random.default_rng(0)
    for c in range(n_classes):
        idx = np.where(y_np == c)[0]
        if len(idx) == 0:
            continue
        rng.shuffle(idx)
        n_train = max(1, int(0.2 * len(idx)))
        n_val = max(1, int(0.2 * len(idx)))
        train_mask[idx[:n_train]] = True
        val_mask[idx[n_train : n_train + n_val]] = True
    test_mask = ~(train_mask | val_mask)
    print(
        f"  train: {int(train_mask.sum().item())}, "
        f"val: {int(val_mask.sum().item())}, "
        f"test: {int(test_mask.sum().item())}"
    )

    return {
        "X": X,
        "A": A,
        "y": y,
        "A_norm": A_norm,
        "A_hat": A_hat,
        "X_np": X_np,
        "A_np": A_np,
        "y_np": y_np,
        "edge_index_np": edge_index_np,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "N": N,
        "F_dim": F_dim,
        "n_classes": n_classes,
        "n_edges_undirected": n_edges_undirected,
        "dataset_name": dataset_name,
    }


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_gnns.db"
    registry_db = "sqlite:///mlfp05_gnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_gnns", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS
# ════════════════════════════════════════════════════════════════════════


def train_node_classifier(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Train a GNN for node classification and log metrics to ExperimentTracker.

    Model selection uses VALIDATION accuracy only: when training ends, the
    weights from the epoch with the highest validation accuracy are
    restored, so later embeddings, plots and registered artifacts describe
    the model whose numbers are reported. Test accuracy is recorded every
    epoch for plotting, but the honest test figure is the one at the
    best-validation epoch: ``test_accs[int(np.argmax(val_accs))]``.

    Returns:
        train_losses: per-epoch training loss
        val_accs: per-epoch validation accuracy
        test_accs: per-epoch test accuracy
    """
    return asyncio.run(
        _train_node_classifier_async(
            model,
            name,
            forward_arg,
            graph_data,
            tracker,
            exp_name,
            epochs,
            lr,
            weight_decay,
        )
    )


async def _train_node_classifier_async(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Async core — uses the kailash-ml 1.1.1 ``tracker.track(...)`` context manager."""
    X = graph_data["X"]
    y = graph_data["y"]
    train_mask = graph_data["train_mask"]
    val_mask = graph_data["val_mask"]
    test_mask = graph_data["test_mask"]
    N = graph_data["N"]
    n_edges = graph_data["n_edges_undirected"]
    dataset_name = graph_data["dataset_name"]
    hidden_dim = 16 if dataset_name == "Karate Club" else 64

    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  [{name}] parameters: {n_params:,}")

    train_losses: list[float] = []
    val_accs: list[float] = []
    test_accs: list[float] = []
    best_val_acc = -1.0
    best_epoch = 0
    best_state: dict[str, torch.Tensor] = {}

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(hidden_dim),
                "epochs": str(epochs),
                "lr": str(lr),
                "weight_decay": str(weight_decay),
                "n_params": str(n_params),
                "dataset": dataset_name,
                "n_nodes": str(N),
                "n_edges": str(n_edges),
            }
        )

        for epoch in range(epochs):
            model.train()
            opt.zero_grad()
            logits = model(X, forward_arg)
            loss = F.cross_entropy(logits[train_mask], y[train_mask])
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

            model.eval()
            with torch.no_grad():
                preds = model(X, forward_arg).argmax(dim=-1)
                v_acc = (preds[val_mask] == y[val_mask]).float().mean().item()
                t_acc = (preds[test_mask] == y[test_mask]).float().mean().item()
            val_accs.append(v_acc)
            test_accs.append(t_acc)
            if v_acc > best_val_acc:  # first epoch wins ties, like np.argmax
                best_val_acc, best_epoch = v_acc, epoch
                best_state = copy.deepcopy(model.state_dict())

            await run.log_metrics(
                {
                    "train_loss": loss.item(),
                    "val_accuracy": v_acc,
                    "test_accuracy": t_acc,
                },
                step=epoch + 1,
            )

            if (epoch + 1) % 25 == 0:
                print(
                    f"  [{name}] epoch {epoch+1:3d}  "
                    f"loss={loss.item():.4f}  val_acc={v_acc:.3f}  test_acc={t_acc:.3f}"
                )

        model.load_state_dict(best_state)
        await run.log_metrics(
            {
                "final_loss": train_losses[-1],
                "final_val_accuracy": val_accs[-1],
                "final_test_accuracy": test_accs[-1],
                "best_val_accuracy": best_val_acc,
                "best_val_epoch": float(best_epoch + 1),
                "test_accuracy_at_best_val": test_accs[best_epoch],
            }
        )
    print(
        f"  [{name}] best val_acc={best_val_acc:.3f} at epoch {best_epoch + 1}; "
        "restored those weights"
    )

    return train_losses, val_accs, test_accs


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════

# ModelVisualizer carries a kailash-ml P2 experimental notice; the canonical
# factory in shared.mlfp05 acknowledges it at the single construction site.

viz = create_visualizer()


def plot_training_curves(
    metrics_dict: dict[str, list[float]],
    title: str,
    y_label: str,
    filename: str,
) -> None:
    """Plot overlaid training curves and save as HTML."""
    fig = viz.training_history(
        metrics=metrics_dict,
        x_label="Epoch",
        y_label=y_label,
    )
    fig.update_layout(title=title)
    filepath = OUTPUT_DIR / filename
    fig.write_html(str(filepath))
    print(f"  Saved: {filepath}")


def plot_node_embeddings(
    embeddings: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
) -> None:
    """2-D PCA projection of node embeddings coloured by class label.

    Uses SVD-based PCA (no sklearn dependency). Nodes of the same class
    should cluster together if the GNN learned meaningful representations.
    """
    emb_centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    Vt = np.linalg.svd(emb_centered, full_matrices=False)[2]
    coords = emb_centered @ Vt.T[:, :2]

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    cmap = plt.get_cmap("tab10", n_classes)
    for c in range(n_classes):
        mask = labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=15,
            alpha=0.6,
            label=f"Class {c}",
        )
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")

    # Text summary of first 3 nodes per class
    print(f"\n  {title} — first 3 nodes per class:")
    for c in range(min(n_classes, 7)):
        rows = coords[labels == c][:3]
        if len(rows) == 0:
            continue
        pretty = ", ".join(f"({r[0]:+.2f}, {r[1]:+.2f})" for r in rows)
        print(f"    class {c}: {pretty}")

    return coords


def plot_graph_with_embeddings(
    A_np: np.ndarray,
    embeddings_2d: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
    max_nodes: int = 200,
) -> None:
    """Draw graph edges on the 2-D embedding space, coloured by class.

    Subsamples to max_nodes for readability on large graphs.
    """
    N = A_np.shape[0]
    if N > max_nodes:
        rng = np.random.default_rng(42)
        subset = rng.choice(N, max_nodes, replace=False)
    else:
        subset = np.arange(N)

    coords = embeddings_2d[subset]
    sub_labels = labels[subset]
    sub_A = A_np[np.ix_(subset, subset)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 9))
    cmap = plt.get_cmap("tab10", n_classes)

    # Draw edges first (behind nodes)
    src_idx, dst_idx = np.where(np.triu(sub_A) > 0)
    for s, d in zip(src_idx, dst_idx):
        ax.plot(
            [coords[s, 0], coords[d, 0]],
            [coords[s, 1], coords[d, 1]],
            color="gray",
            alpha=0.08,
            linewidth=0.5,
        )

    # Draw nodes
    for c in range(n_classes):
        mask = sub_labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=25,
            alpha=0.7,
            label=f"Class {c}",
            zorder=2,
        )

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


def plot_attention_weights(
    alpha: np.ndarray,
    A_np: np.ndarray,
    labels: np.ndarray,
    title: str,
    filename: str,
    node_idx: int = 0,
    top_k: int = 10,
) -> None:
    """Visualise attention weights for a single node's neighbourhood.

    Shows which neighbours the GAT layer attends to most strongly.
    """
    neighbours = np.where(A_np[node_idx] > 0)[0]
    if len(neighbours) == 0:
        print(f"  Node {node_idx} has no neighbours — skipping attention plot")
        return

    weights = alpha[node_idx, neighbours]
    order = np.argsort(-weights)[:top_k]
    top_neighbours = neighbours[order]
    top_weights = weights[order]

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    bar_colors = [plt.get_cmap("tab10")(labels[n] % 10) for n in top_neighbours]
    ax.barh(
        range(len(top_neighbours)),
        top_weights,
        color=bar_colors,
        edgecolor="white",
    )
    ax.set_yticks(range(len(top_neighbours)))
    ax.set_yticklabels(
        [f"Node {n} (class {labels[n]})" for n in top_neighbours],
        fontsize=9,
    )
    ax.set_xlabel("Attention Weight")
    ax.set_title(
        f"{title}\nNode {node_idx} (class {labels[node_idx]}) attending to top-{top_k} neighbours",
        fontsize=12,
        fontweight="bold",
    )
    ax.invert_yaxis()
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Register a model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metrics,
    )
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Synchronously register a model."""
    return asyncio.run(_register_model(registry, name, model, metrics))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 6.3: GraphSAGE (Sample and Aggregate)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Why GCN doesn't scale to large graphs (full adjacency in memory)
#   - Inductive learning: why GraphSAGE CAN embed unseen nodes (and why
#     this Cora exercise does not yet prove it)
#   - Neighbour sampling strategy: fixed-size random subsets per node
#   - Separate self/neighbour projections for richer representations
#   - Train a scalable node classifier on the Cora citation network
#   - Track training with kailash-ml ExperimentTracker
#
# PREREQUISITES: M5/ex_6.1 (GCN), M5/ex_6.2 (GAT).
# ESTIMATED TIME: ~30 min
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash_ml.types import MetricSpec

import matplotlib.pyplot as plt



## PHASE 1 — THEORY: Why GCN Doesn't Scale


WHY GCN DOESN'T SCALE:
  - GCN needs full adjacency matrix in memory: O(N^2) for dense, O(E) for sparse
  - Cora (2.7K nodes): fine. Real graphs (500K+ nodes): memory explosion
  - Multi-hop aggregation expands exponentially: 2 hops of degree-50 = 2,500 nodes

  GRAPHSAGE'S THREE KEY IDEAS:

  1. SAMPLE: randomly pick K neighbours per node (not ALL)
     -> Fixed memory budget regardless of graph size
     -> Like dropout: different samples each epoch = regularisation

  2. AGGREGATE: learnable function over sampled neighbours
     -> Mean, LSTM, or pooling aggregator
     -> A FUNCTION, not a lookup — works on any neighbour set

  3. INDUCTIVE: learns HOW to aggregate, not WHAT to embed
     -> New nodes at inference time? Sample their neighbours and run
        the learned aggregator
     -> GCN as trained in ex_6.1 is used TRANSDUCTIVELY: one fixed,
        full-graph normalised adjacency, with the test nodes already in
        the graph during training. (GAT also learns a function of node
        features and was shown to be inductive in its own paper.)

  Formula: h'_i = sigma( W_self @ h_i + W_neigh @ MEAN(sample(N(i))) )
  Separate W_self and W_neigh = "what I know" vs "what neighbours say"


In [ ]:
#
# GCN and GAT both require the FULL adjacency matrix during training.
# For Cora (2,708 nodes), that's a 2708 x 2708 matrix — no problem.
# But real-world graphs are much bigger:
#
#   - A city-scale food delivery network: ~500K users x ~50K restaurants
#   - A global social network: billions of user nodes
#   - A web-scale knowledge graph: hundreds of billions of facts (edges)
#
# A 500K x 500K dense adjacency matrix needs ~1 TB of memory. Even
# sparse representations strain GPU memory when you need multi-hop
# neighbourhood aggregation.
#
# GraphSAGE (SAmple and aggrEGATE) solves this with three key ideas:
#
# 1. SAMPLE: Instead of aggregating ALL neighbours, randomly sample
#    a fixed number K (e.g., 10) per node. This bounds memory usage
#    regardless of graph size.
#
# 2. AGGREGATE: Use a learnable aggregator (mean, LSTM, or pooling)
#    over the sampled neighbours. The aggregator is a FUNCTION, not
#    a lookup table — it works on any set of neighbours.
#
# 3. INDUCTIVE: Because GraphSAGE learns an aggregation FUNCTION
#    rather than per-node embeddings, it can generalise to nodes it
#    has never seen during training. A new restaurant added to the
#    platform can be classified immediately using its neighbours.
#
# The formula:
#   h'_i = sigma( W_self @ h_i + W_neigh @ MEAN(h_j for j in Sample(N(i))) )
#
# Notice: SEPARATE weight matrices for self (W_self) and neighbours
# (W_neigh). This lets the model learn different transformations for
# "what I know about myself" vs "what my neighbours tell me".
print("=" * 70)
print("  PHASE 1 — THEORY: GraphSAGE — Sample and Aggregate")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: GraphSAGE Layer Implementation


During training: randomly sample at most K neighbours per node
    (regularisation + scalability). At eval: use all neighbours for
    deterministic output (like dropout).

    Separate W_self and W_neigh allow the model to learn different
    transformations for a node's own features vs its neighbours'.


Two-layer GraphSAGE for node classification.


Return the hidden-layer embedding (before classification head).


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: GraphSAGE Layer + Model")
print("=" * 70)

# Load graph data and set up engines
graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
y = graph_data["y"]
y_np = graph_data["y_np"]
A_np = graph_data["A_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
EPOCHS = 100
SAMPLE_K = 10  # Max neighbours to sample per node


class GraphSAGELayer(nn.Module):

    def __init__(self, in_dim: int, out_dim: int, sample_k: int = 10):
        super().__init__()
        self.W_self = nn.Linear(in_dim, out_dim, bias=False)
        self.W_neigh = nn.Linear(in_dim, out_dim, bias=False)
        self.sample_k = sample_k

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        n = h.size(0)

        # Neighbour sampling: for each node, keep at most sample_k neighbours
        # by zeroing out excess connections. At eval time, use all neighbours
        # for deterministic output (like dropout).
        if self.training and self.sample_k < n:
            sample_mask = torch.zeros_like(adj)
            for i in range(n):
                neigh_idx = torch.where(adj[i] > 0)[0]
                if len(neigh_idx) <= self.sample_k:
                    sample_mask[i, neigh_idx] = 1.0
                else:
                    perm = torch.randperm(len(neigh_idx), device=h.device)[
                        : self.sample_k
                    ]
                    sample_mask[i, neigh_idx[perm]] = 1.0
            adj_sampled = sample_mask
        else:
            adj_sampled = adj

        # Mean aggregation: average the features of sampled neighbours
        deg_sampled = adj_sampled.sum(dim=1, keepdim=True).clamp(min=1.0)
        h_neigh = (adj_sampled @ h) / deg_sampled  # (N, in_dim)

        # Combine self and neighbour representations
        h_self = self.W_self(h)
        h_agg = self.W_neigh(h_neigh)
        return h_self + h_agg  # additive combination


class GraphSAGE(nn.Module):

    def __init__(
        self, in_dim: int, hidden_dim: int, n_classes: int, sample_k: int = 10
    ):
        super().__init__()
        self.l1 = GraphSAGELayer(in_dim, hidden_dim, sample_k=sample_k)
        self.l2 = GraphSAGELayer(hidden_dim, n_classes, sample_k=sample_k)

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.l1(h, adj))
        h = F.dropout(h, p=0.5, training=self.training)
        return self.l2(h, adj)

    def embed(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        return F.relu(self.l1(h, adj))


sage = GraphSAGE(
    in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes, sample_k=SAMPLE_K
)
n_params = sum(p.numel() for p in sage.parameters())
print(f"\n  GraphSAGE architecture:")
print(f"    Layer 1: GraphSAGELayer({F_dim} -> {HIDDEN_DIM}, sample_k={SAMPLE_K})")
print(f"    Layer 2: GraphSAGELayer({HIDDEN_DIM} -> {n_classes}, sample_k={SAMPLE_K})")
print(f"    Total parameters: {n_params:,}")
print(f"\n  How sampling works:")
print(f"    Training: each node samples up to {SAMPLE_K} neighbours per epoch")
print(f"    Eval: use all neighbours (deterministic, like dropout)")
print(f"    Effect: regularisation + bounded memory per mini-batch")

# Show how sampling affects neighbourhood size
degrees = A_np.sum(axis=1)
nodes_needing_sample = (degrees > SAMPLE_K).sum()
print(f"\n  Sampling impact on {dataset_name}:")
print(f"    Avg degree: {degrees.mean():.1f}")
print(f"    Max degree: {int(degrees.max())}")
print(f"    Nodes with degree > {SAMPLE_K}: {nodes_needing_sample} / {N}")
print(f"    -> {nodes_needing_sample} nodes will have sampled neighbourhoods")



### Build Checkpoint


In [ ]:
assert isinstance(sage, nn.Module), "GraphSAGE should be an nn.Module"
assert n_params > 0, "GraphSAGE should have learnable parameters"
print("\n--- Build checkpoint passed --- GraphSAGE architecture created\n")



## PHASE 3 — TRAIN: Node Classification on Cora


In [ ]:
print("=" * 70)
print(f"  PHASE 3 — TRAIN: GraphSAGE on {dataset_name}")
print("=" * 70)

sage_losses, sage_val, sage_test = train_node_classifier(
    model=sage,
    name="GraphSAGE",
    forward_arg=A,
    graph_data=graph_data,
    tracker=tracker,
    exp_name=exp_name,
    epochs=EPOCHS,
)



### Train Checkpoint


In [ ]:
assert len(sage_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses for GraphSAGE"
assert sage_losses[-1] < sage_losses[0], "GraphSAGE loss should decrease"
# Model selection by VALIDATION accuracy; report test accuracy at that
# epoch (the harness has already restored that epoch's weights).
best_epoch = int(np.argmax(sage_val))
best_val = sage_val[best_epoch]
best_test = sage_test[best_epoch]
print(f"\n  GraphSAGE Results:")
print(f"    Best validation accuracy: {best_val:.4f} (epoch {best_epoch + 1})")
print(f"    Test accuracy, that epoch: {best_test:.4f}")
print(f"    Final loss:               {sage_losses[-1]:.4f}")
# INTERPRETATION: GraphSAGE is designed to be INDUCTIVE — it learns an
# aggregation FUNCTION that can be applied to nodes it never saw. Note
# that this run is still transductive: every Cora node, including the
# test nodes, sits in the graph during training. During training, it
# randomly samples K neighbours per node (like dropout for graphs),
# which provides regularisation and makes it scalable to large graphs.
# The separate W_self and W_neigh projections let the model learn
# different transformations for a node's own features versus its
# neighbours' features.
print("\n--- Train checkpoint passed --- GraphSAGE trained successfully\n")



## DIAGNOSTIC CHECKPOINT — Prescription Pad before Visualise


In [ ]:
# run_diagnostic_checkpoint instruments the trained model, replays a few
# forward/backward passes of the REAL training objective (cross-entropy
# on the labelled training nodes; no weights are updated) and replays
# the per-epoch training losses. The whole graph is one "batch", so the
# loader is the same full-graph tuple repeated.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _node_loss(m, batch):
    feats, graph, labels, mask = batch
    return F.cross_entropy(m(feats, graph)[mask], labels[mask])


diag, findings = run_diagnostic_checkpoint(
    sage,
    [(X, A, y, graph_data["train_mask"])] * 4,
    _node_loss,
    title="GraphSAGE — Sample and Aggregate",
    n_batches=4,
    train_losses=sage_losses,
    show=False,
)
print_prescription_pad(findings, "GraphSAGE — Sample and Aggregate")
# HOW TO READ IT (your readings depend on your run):
#  GRADIENT FLOW — a 2-layer GNN rarely vanishes. Exploding readings
#     usually mean the propagation matrix is not normalised (a raw
#     adjacency multiplies feature scale by node degree) or the learning
#     rate is too high.
#  DEAD NEURONS — this model applies its activation functionally
#     (F.relu / F.elu), so there is no activation LAYER for the
#     instrument to hook; an UNKNOWN reading here is expected, not a
#     fault. Use nn.ReLU modules if you want this reading.
#  LOSS TREND — this sees only the training loss. Over-fitting shows up
#     in the gap between the validation and training curves, not here.



## PHASE 4 — VISUALISE: Embeddings + Sampling Effect


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: GraphSAGE Embeddings + Sampling Analysis")
print("=" * 70)

sage.eval()
with torch.no_grad():
    sage_emb = sage.embed(X, A).cpu().numpy()

# Plot 1: 2-D PCA of node embeddings
coords = plot_node_embeddings(
    embeddings=sage_emb,
    labels=y_np,
    n_classes=n_classes,
    title=f"GraphSAGE Node Embeddings — {dataset_name}",
    filename="sage_node_embeddings.png",
)

# Plot 2: Graph structure on embedding space
plot_graph_with_embeddings(
    A_np=A_np,
    embeddings_2d=coords,
    labels=y_np,
    n_classes=n_classes,
    title=f"GraphSAGE — Graph Structure in Embedding Space ({dataset_name})",
    filename="sage_graph_embeddings.png",
)

# Plot 3: Training curves
plot_training_curves(
    metrics_dict={"GraphSAGE train loss": sage_losses},
    title="GraphSAGE Training Loss",
    y_label="Cross-Entropy Loss",
    filename="sage_loss_curve.html",
)
plot_training_curves(
    metrics_dict={
        "GraphSAGE val accuracy": sage_val,
        "GraphSAGE test accuracy": sage_test,
    },
    title="GraphSAGE Accuracy",
    y_label="Accuracy",
    filename="sage_accuracy_curves.html",
)

# Plot 4: Analyse the effect of sampling K on embedding variance
# Run multiple forward passes in training mode to show stochastic embeddings
print("\n  Sampling stochasticity analysis:")
embedding_variances = []
sage.train()  # Enable sampling
with torch.no_grad():
    embeddings_list = []
    for trial in range(5):
        emb_trial = sage.embed(X, A).cpu().numpy()
        embeddings_list.append(emb_trial)
    embeddings_stack = np.stack(embeddings_list)  # (5, N, hidden)
    per_node_var = embeddings_stack.var(axis=0).mean(axis=1)  # (N,)

sage.eval()  # Restore eval mode

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of per-node embedding variance
axes[0].hist(per_node_var, bins=50, color="coral", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Embedding Variance Across Samples", fontsize=11)
axes[0].set_ylabel("Number of Nodes", fontsize=11)
axes[0].set_title(
    "Stochastic Embedding Variance\n(5 forward passes with sampling)", fontsize=12
)

# Right: variance vs degree
axes[1].scatter(degrees, per_node_var, s=8, alpha=0.4, color="steelblue")
axes[1].set_xlabel("Node Degree", fontsize=11)
axes[1].set_ylabel("Embedding Variance", fontsize=11)
axes[1].set_title(
    f"Variance vs Degree (sample_k={SAMPLE_K})\nHigher degree -> more sampling -> more variance",
    fontsize=12,
)

plt.tight_layout()
filepath = OUTPUT_DIR / "sage_sampling_variance.png"
plt.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

high_var = per_node_var > np.percentile(per_node_var, 90)
low_var = per_node_var <= np.percentile(per_node_var, 10)
print(
    f"    High-variance nodes (top 10%): {int(high_var.sum())}, "
    f"mean degree {degrees[high_var].mean():.1f}"
)
print(
    f"    Low-variance nodes (bottom 10%): {int(low_var.sum())}, "
    f"mean degree {degrees[low_var].mean():.1f}"
)
small = degrees <= SAMPLE_K
print(
    f"    Nodes with degree <= {SAMPLE_K}: their layer-1 neighbourhood is never "
    f"subsampled (mean variance {per_node_var[small].mean():.2e} vs "
    f"{per_node_var[~small].mean():.2e} for larger-degree nodes)"
)



### Visualise Checkpoint


In [ ]:
assert sage_emb.shape == (
    N,
    HIDDEN_DIM,
), f"Embedding shape should be ({N}, {HIDDEN_DIM})"
print("\n--- Visualise checkpoint passed --- GraphSAGE embeddings + variance plotted\n")



## PHASE 5 — APPLY: Recommendation Engine for Food Delivery


SCENARIO (illustrative): You're building a recommendation engine for a
  Singapore food delivery platform.

  THE GRAPH:
  - User nodes: ~500K users with features (location, order frequency, cuisine prefs)
  - Restaurant nodes: ~50K restaurants (cuisine type, price range, rating)
  - Edges: user-restaurant orders (weighted by frequency)
  - Bipartite graph: users only connect to restaurants, not to each other

  WHY GRAPHSAGE IS THE RIGHT CHOICE:
  1. SCALE: 550K nodes = GCN's adjacency matrix would need 302 billion entries
     GraphSAGE samples 10 neighbours per node = bounded memory
  2. INDUCTIVE: new restaurants join daily. A full-graph GCN like ex_6.1's
     has to be re-run (and usually retrained) on the whole updated graph.
     GraphSAGE is trained to embed a new restaurant directly from a
     sample of its first customers' features.
  3. COLD START: a new restaurant with just 3 orders can be embedded —
     GraphSAGE aggregates those 3 users. Whether that embedding is GOOD
     is something you measure on held-out new restaurants, not assume.

  RECOMMENDATION PIPELINE:
  1. Train GraphSAGE on the user-restaurant graph
  2. Each user gets an embedding (captures taste preferences via neighbours)
  3. Each restaurant gets an embedding (captures customer base profile)
  4. Score = dot_product(user_embedding, restaurant_embedding)
  5. Rank restaurants by score for each user -> personalised recommendations


HOW GRAPHSAGE DIFFERS FROM THE NEIGHBOUR-MAJORITY BASELINE:
  - CF just counts neighbours — GraphSAGE LEARNS what to aggregate
  - CF has no features — GraphSAGE combines structure with node features
  - CF is one-hop — 2-layer GraphSAGE captures 2-hop patterns
  - CF is fixed — GraphSAGE adapts its aggregation during training

  DEPLOYMENT CONSIDERATIONS:
  1. Mini-batch training: sample subgraphs, not full graph (torch_geometric)
  2. Pre-compute embeddings offline; serve recommendations from cache
  3. Retrain weekly with new orders; update embeddings for active users daily
  4. Track recommendation quality with ExperimentTracker (CTR, order rate)
  5. A/B test: GraphSAGE recs vs popularity-based recs vs CF baseline


In [ ]:
print("=" * 70)
print("  PHASE 5 — APPLY: Food Delivery Recommendations")
print("=" * 70)
print(
)

# Demonstrate a neighbour-majority (CF-style) baseline vs GraphSAGE
# Using Cora as proxy: predict class membership from neighbourhood
print("  Neighbour-Majority Baseline vs GraphSAGE:")

# Baseline: predict a node's class by majority vote over the labels of its
# TRAINING-set neighbours only. Using every neighbour's label would read
# the true labels of validation/test nodes — label leakage.
train_mask = graph_data["train_mask"]
test_mask = graph_data["test_mask"]
fallback_class = int(torch.bincount(y[train_mask], minlength=n_classes).argmax())
majority_preds = torch.full((N,), fallback_class, dtype=torch.long, device=device)

for i in range(N):
    neighbours = torch.where((A[i] > 0) & train_mask)[0]
    if len(neighbours) == 0:
        continue  # no labelled neighbour: keep the majority training class
    neighbour_labels = y[neighbours]
    # Majority vote
    counts = torch.bincount(neighbour_labels, minlength=n_classes)
    majority_preds[i] = counts.argmax()

cf_acc = (majority_preds[test_mask] == y[test_mask]).float().mean().item()

print(f"    Neighbour majority (train labels only): {cf_acc:.4f}")
print(f"    GraphSAGE (learned aggregation):        {best_test:.4f}")
improvement = best_test - cf_acc
print(
    f"    Difference:                             {improvement:+.4f} ({improvement*100:+.1f} pp)"
)

print(
)

# Register the GraphSAGE model
if has_registry:
    version = register_model(
        registry=registry,
        name=f"m5_graphsage_{dataset_name.lower().replace(' ', '_')}",
        model=sage,
        metrics=[
            MetricSpec(name="best_val_accuracy", value=best_val),
            MetricSpec(name="test_accuracy_at_best_val", value=best_test),
            MetricSpec(name="final_loss", value=sage_losses[-1]),
            MetricSpec(name="cf_baseline_accuracy", value=cf_acc),
            MetricSpec(name="improvement_over_cf", value=improvement),
            MetricSpec(name="sample_k", value=float(SAMPLE_K)),
            MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered GraphSAGE: version={version.version}, val_acc={best_val:.4f}")



### Apply Checkpoint


In [ ]:
assert cf_acc > 0.0, "CF baseline should produce non-zero accuracy"
print("\n--- Apply checkpoint passed --- recommendation scenario demonstrated\n")



## REFLECTION


GRAPHSAGE (Hamilton, Ying & Leskovec, 2017):
  [x] Neighbour sampling: fixed K neighbours per node bounds memory
  [x] Mean aggregator: MEAN(h_j for j in Sample(N(i)))
  [x] Separate projections: W_self @ h_i + W_neigh @ h_agg
  [x] INDUCTIVE design: a learned aggregation FUNCTION can embed unseen
      nodes — this run trained on the full Cora graph, so it shows the
      mechanism; proving generalisation needs held-out nodes
  [x] Trained on {dataset_name}: {best_val:.1%} val accuracy, {best_test:.1%} test accuracy
  [x] Analysed sampling stochasticity: variance vs node degree
  [x] Compared with a neighbour-majority baseline: {improvement*100:+.1f} percentage points

  THREE-WAY COMPARISON (so far):
  - GCN: fixed weights, full graph, fast, simple
  - GAT: learned attention, full graph, interpretable
  - GraphSAGE: sampling + learned aggregation, SCALABLE, INDUCTIVE

  WHEN TO USE GRAPHSAGE:
  - Graph has 100K+ nodes (GCN/GAT memory-bound)
  - New nodes arrive at inference time (inductive requirement)
  - Mini-batch training needed (can't fit full graph on GPU)
  - Recommendation systems, dynamic social networks, evolving knowledge graphs

  Next: Exercise 6.4 — Link Prediction: predict missing edges with a
  dot-product decoder (foundation of recommendation systems)...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — GraphSAGE")
print("=" * 70)
print(
)

# Clean up
await conn.close()

